# Natural Language Processing INM343
### Comparative Legal Clause Classification



#### The notebook compares four families of approaches:

- dummy baselines for lower-bound context
- classical sparse-text models using TF-IDF features
- an optional fine-tuned transformer classifier
- an optional Qwen2.5-Instruct prompting baseline



**normal exhaustive all-variants mode:** this notebook expects the project `modules/` folder to be present at the repository root and enables every module-exposed experiment family/variant: classical tokenizer/model grids, BiLSTM, configured transformers, transformer HPT, Qwen prompting variants, instruction-tuned LLM variants, agentic review, and CUAD external evaluation.


This version runs all module-exposed variants in the normal master notebook structure. It is intended as the single source-of-truth notebook for exhaustive reruns.

All stages now support separate W&B runs from the single master notebook when `WANDB_RUN_STRATEGY = "per_stage"` and `WANDB_ENABLED_STAGES = "all"`.


## 1. Colab Setup, Imports, and Configuration

This stage prepares the runtime so the same notebook can run locally or in Google Colab. In Colab, upload, unzip, sync, or clone the whole project folder, not just this notebook.



Importing Libraries and Modules

In [1]:
# @title
from pathlib import Path

import importlib.util
import os
import subprocess
import sys
import numpy as np
import pandas as pd
import pickle

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import json
import math
import re

import pandas as pd
from datetime import datetime, timezone
from IPython.display import display
import shutil


import gc


File Setup

In [2]:
# @title
PROJECT_ROOT_OVERRIDE = os.environ.get("LEDGAR_PROJECT_ROOT", "").strip()
AUTO_MOUNT_GOOGLE_DRIVE = True
INSTALL_REQUIREMENTS_IN_COLAB = True


def running_in_colab() -> bool:
    return "COLAB_RELEASE_TAG" in os.environ or importlib.util.find_spec("google.colab") is not None


IN_COLAB = running_in_colab()

In [3]:
# @title

if IN_COLAB:
    print("Google Colab runtime detected.")

if IN_COLAB and AUTO_MOUNT_GOOGLE_DRIVE:
    try:
        if Path("/content/drive/MyDrive").exists():
            print("Google Drive is already available.")
        else:
            from google.colab import drive

            drive.mount("/content/drive")
    except Exception as exc:
        print(f"Google Drive mount skipped/failed: {type(exc).__name__}: {exc}")


def looks_like_project_root(path: Path) -> bool:
    return (path / "pyproject.toml").exists() and (path / "modules").is_dir()


def project_root_candidates_near(path: Path) -> list[Path]:
    path = path.expanduser()
    candidates = [path, *path.parents]
    if path.exists() and path.is_dir():
        for pattern in (
            "pyproject.toml",
            "*/pyproject.toml",
            "*/*/pyproject.toml",
            "*/*/*/pyproject.toml",
        ):
            candidates.extend(pyproject.parent for pyproject in path.glob(pattern))
    deduped = []
    seen = set()
    for candidate in candidates:
        try:
            resolved = candidate.resolve()
        except Exception:
            resolved = candidate
        if resolved not in seen:
            deduped.append(resolved)
            seen.add(resolved)
    return deduped


def parent_search(start: Path) -> Path | None:
    for candidate in project_root_candidates_near(start):
        if looks_like_project_root(candidate):
            return candidate
    return None


def common_colab_candidates() -> list[Path]:
    candidates = [
        Path("/content/drive/MyDrive/Colab Notebooks/Education/NLP/Natural-Language-Processing"),
    ]
    for base in (Path("/content"), Path("/content/drive/MyDrive")):
        if base.exists():
            for pattern in (
                "Natural-Language-Processing",
                "*/Natural-Language-Processing",
                "*/*/Natural-Language-Processing",
                "*/*/*/Natural-Language-Processing",
            ):
                candidates.extend(base.glob(pattern))
    return candidates


def find_notebook_project_root() -> Path:
    if PROJECT_ROOT_OVERRIDE:
        override = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
        for candidate in project_root_candidates_near(override):
            if looks_like_project_root(candidate):
                if candidate != override:
                    print(f"PROJECT_ROOT_OVERRIDE pointed to a parent folder; using nested project root: {candidate}")
                return candidate
        raise FileNotFoundError(
            f"PROJECT_ROOT_OVERRIDE does not contain pyproject.toml and modules/, and no nested project root was found under it: {override}\n"
            "Check the Drive folder path, or run this diagnostic: list(Path('/content/drive/MyDrive').glob('**/pyproject.toml'))"
        )

    root = parent_search(Path.cwd())
    if root is not None:
        return root

    if IN_COLAB:
        for candidate in common_colab_candidates():
            if candidate.exists() and looks_like_project_root(candidate):
                return candidate.resolve()

    raise FileNotFoundError(
        "Could not find the project root containing pyproject.toml and modules/.\n"
        "In Colab, upload or clone the whole repository, then set PROJECT_ROOT_OVERRIDE "
        "near the top of this cell to that folder. Current working directory: "
        f"{Path.cwd()}"
    )

In [4]:
# @title
# Find the project root and add it to sys.path so that imports work, even if the notebook is opened in a subfolder or outside the project.
PROJECT_ROOT = find_notebook_project_root()
os.environ["LEDGAR_PROJECT_ROOT"] = str(PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# List of (import_name, pip_name) for packages commonly used in notebooks. pip_name can be None if it's the same as import_name.
REQUIRED_NOTEBOOK_PACKAGES = [
    ("pandas", "pandas"),
    ("datasets", "datasets"),
    ("huggingface_hub", "huggingface_hub"),
    ("sklearn", "scikit-learn"),
    ("joblib", "joblib"),
    ("matplotlib", "matplotlib"),
    ("torch", "torch"),
    ("transformers", "transformers"),
    ("accelerate", "accelerate"),
    ("optuna", "optuna>=3.6"),
]

# In Colab, install all requirements from requirements-colab.txt if any are missing, to avoid multiple pip installs.
def ensure_notebook_package(import_name: str, pip_name: str | None = None) -> None:
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or import_name])

# Check for missing imports before installing requirements in Colab, to avoid unnecessary pip installs and speed up notebook startup.
missing_imports = [name for name, _ in REQUIRED_NOTEBOOK_PACKAGES if importlib.util.find_spec(name) is None]
requirements_path = PROJECT_ROOT / "requirements-colab.txt"


if IN_COLAB and INSTALL_REQUIREMENTS_IN_COLAB and requirements_path.exists() and missing_imports:
    print(f"Installing Colab requirements from {requirements_path}.")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_path)])
    except subprocess.CalledProcessError as e:
        print(f"WARNING: Installation from requirements-colab.txt failed: {e}")
        print("Attempting to install individual required packages instead.")
        for import_name, pip_name in REQUIRED_NOTEBOOK_PACKAGES:
            try:
                ensure_notebook_package(import_name, pip_name)
            except subprocess.CalledProcessError as sub_e:
                print(f"ERROR: Failed to install individual package {pip_name or import_name}: {sub_e}")
else:
    for import_name, pip_name in REQUIRED_NOTEBOOK_PACKAGES:
        ensure_notebook_package(import_name, pip_name)

# Explicitly import optuna now that installation is ensured
try:
    import optuna
    print(f"Optuna version: {optuna.__version__}")
except ImportError:
    print("Optuna could not be imported. HPT stages may fail.")

FileNotFoundError: Could not find the project root containing pyproject.toml and modules/.
In Colab, upload or clone the whole repository, then set PROJECT_ROOT_OVERRIDE near the top of this cell to that folder. Current working directory: C:\Users\ybenj

Custom Modules and Libraries

In [ ]:
# @title
from modules.data_setup import (
    adapt_cuad_to_clause_classification,
    build_project_paths,
    download_cuad_if_missing,
    load_cuad_raw_files,
    load_or_download_ledgar,
    normalise_whitespace,
    print_dataset_availability,
    seed_everything,
)
from modules.preprocessing import (
    bpe_encode_text,
    bpe_encode_word,
    clean_html_entities,
    corpus_word_frequencies,
    create_ledgar_eda,
    legal_safe_tokenise,
    negation_aware_tokenise,
    preprocess_ledgar,
    preprocessing_technique_rundown,
    regex_tokenise,
    train_bpe_tokeniser,
    write_preprocessing_rundown,
)
from modules.baselines import run_baseline_experiments
from modules.classical_models import run_classical_experiments
from modules.sequence_model import SequenceModelConfig, train_sequence_classifier
from modules.transformer_model import train_transformer_classifier
from modules.transformer_hpt import TransformerHPTConfig, run_two_stage_transformer_hpt
from modules.qwen_prompting import run_qwen_baseline
from modules.cuad_external import run_cuad_external_evaluation
from modules.agentic_review import run_agentic_review
from modules.llm_evaluation import evaluate_instruction_tuned_llms
from modules.evaluation import save_final_comparison
from modules.error_analysis import run_error_analysis
from modules.wandb_reporting import finish_wandb_run, log_wandb_outputs, start_wandb_run


| Setting | Value | Purpose |
|---|---:|---|
| `RUN_CLASSICAL_MODELS` | `True` | Runs classical TF-IDF variants. |
| `CLASSICAL_TOKENIZER_VARIANTS` | `negation_aware`, `legal_safe` | Runs both tokenizer variants exposed by `modules.classical_models`. |
| `RUN_SEQUENCE_MODEL` | `True` | Runs the BiLSTM/sequence baseline. |
| `RUN_CONFIGURED_TRANSFORMER_BASELINES` | `True` | Runs configured transformer baselines for each transformer model variant. |
| `RUN_TRANSFORMER_HPT` | `True` | Runs two-stage transformer HPT. |
| `TRANSFORMER_MODEL_VARIANTS` | DistilBERT + Legal-BERT + Contracts-BERT | Runs all configured transformer variants. |
| `TRANSFORMER_HPT_MODEL_VARIANTS` | DistilBERT + Legal-BERT + Contracts-BERT | Runs HPT for all transformer variants. |
| `HPT_RANDOM_TRIALS` | `32` | Stage 5A random-search trials per transformer model. |
| `HPT_BAYES_TRIALS` | `32` | Stage 5B Bayesian trials per transformer model. |
| `RUN_QWEN_BASELINE` | `True` | Runs Qwen prompting module. The module itself evaluates zero-shot, static few-shot, and retrieval few-shot. |
| `QWEN_MODEL_VARIANTS` | Qwen 3B + Qwen 7B | Runs both Qwen model variants. |
| `RUN_INSTRUCTION_LLM_EVAL` | `True` | Runs instruction-tuned LLM evaluation with validation decoding search. |
| `INSTRUCTION_LLM_MODEL_KEYS` | SaulLM 7B, Qwen small, Qwen 7B | Runs all instruction LLM variants exposed by the module. |
| `RUN_AGENTIC_EXTENSION` | `True` | Runs the agentic review stage. |
| `RUN_CUAD_EXTERNAL_EVAL` | `True` | Runs CUAD as external/out-of-domain evaluation only. |

This notebook assumes the real project `modules/` folder is present at repo root. It does not fake unavailable variants; it calls the module APIs directly and records skips/failures if a model cannot load.

W&B is configurable from the single master notebook:

- `WANDB_RUN_STRATEGY = "per_stage"` creates a separate W&B run for each notebook stage.
- `WANDB_RUN_STRATEGY = "single"` keeps one W&B run for the whole master notebook.
- `WANDB_RUN_STRATEGY = "disabled"` disables W&B without changing the experiment flags.


In [ ]:
# @title
SEED = 42

DATASET_NAME = "LEDGAR"
TOP_K_LABELS = 20

# Classical model grid: both module tokenizers + all model families exposed by modules.classical_models.
MAX_FEATURES_LIST = [10000, 30000]
NGRAM_RANGES = [(1, 1), (1, 2)]
MIN_DF_LIST = [1, 2, 5]
CLASSICAL_C_VALUES = [0.1, 1.0, 3.0, 10.0]
CLASSICAL_NB_ALPHA_VALUES = [0.1, 0.5, 1.0]
CLASSICAL_TOKENIZER_VARIANTS = ["negation_aware", "legal_safe"]

# Normal exhaustive mode: run every stage/variant exposed by the module folder.
RUN_CLASSICAL_MODELS = True
RUN_NAIVE_BAYES = True
RUN_SEQUENCE_MODEL = True
RUN_CONFIGURED_TRANSFORMER_BASELINES = True
RUN_TRANSFORMER = True
RUN_TRANSFORMER_HPT = True
RUN_SECONDARY_TRANSFORMERS = True
RUN_QWEN_BASELINE = True
RUN_INSTRUCTION_LLM_EVAL = True
RUN_AGENTIC_EXTENSION = True
RUN_CUAD_EXTERNAL_EVAL = True

# No smoke mode and no sample caps. These are intended for normal exhaustive / many-GPU runs.
TRANSFORMER_HPT_SMOKE_TEST = False
HPT_RANDOM_TRIALS = 5
HPT_BAYES_TRIALS = 5
HPT_FINAL_RETRAIN_EPOCHS = 3
HPT_MAX_TRAIN_SAMPLES = None
HPT_MAX_VALIDATION_SAMPLES = None
HPT_MAX_EVAL_SAMPLES = None

RUN_WANDB = True
WANDB_PROJECT = os.environ.get("WANDB_PROJECT", "ledgar-clause-classification")
WANDB_ENTITY = os.environ.get("WANDB_ENTITY", "").strip() or None
WANDB_MODE = os.environ.get("WANDB_MODE", "online")

# W&B run strategy for this single master notebook:
# - "per_stage": create and finish a separate W&B run around each notebook stage.
# - "single": keep one W&B run open for the whole notebook.
# - "disabled": disable W&B without changing the experiment pipeline.
WANDB_RUN_STRATEGY = os.environ.get("WANDB_RUN_STRATEGY", "per_stage").strip().lower()

# Comma-separated stage names or "all". Example:
# "01_setup_config,02_raw_dataset_setup,03_preprocessing_eda,04_shared_result_state,05_dummy_baselines,06_classical_tfidf_all_variants"
WANDB_ENABLED_STAGES = os.environ.get("WANDB_ENABLED_STAGES", "all").strip()
WANDB_STAGE_GROUP = os.environ.get("WANDB_STAGE_GROUP", "ledgar-coursework-all-variants-stages").strip()
WANDB_SINGLE_RUN_NAME = os.environ.get("WANDB_SINGLE_RUN_NAME", "ledgar-all-variants-single-notebook").strip()
WANDB_HPT_USE_INTERNAL_RUNS = False  # False = one visible outer transformer stage run; True = HPT module can create internal trial runs.

WANDB_LOG_ARTIFACTS = True
WANDB_LOG_TEXT_TABLES = False
WANDB_LOG_MODEL_FILES = False

TRANSFORMER_MODEL_NAME = "distilbert-base-uncased"
OPTIONAL_LEGAL_MODEL_NAME = "nlpaueb/legal-bert-base-uncased"
CONTRACTBERT_MODEL_NAME = "nlpaueb/bert-base-uncased-contracts"
TRANSFORMER_MODEL_VARIANTS = [
    TRANSFORMER_MODEL_NAME,
    OPTIONAL_LEGAL_MODEL_NAME,
    CONTRACTBERT_MODEL_NAME,
]
TRANSFORMER_HPT_MODEL_VARIANTS = TRANSFORMER_MODEL_VARIANTS.copy()
MAX_TRANSFORMER_LENGTH = 256

# Qwen prompting module runs zero-shot, static few-shot, and retrieval few-shot for each configured model.
QWEN_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
QWEN_MODEL_VARIANTS = ["Qwen/Qwen2.5-3B-Instruct", "Qwen/Qwen2.5-7B-Instruct"]
QWEN_EVAL_SAMPLE_SIZE = 1_000_000_000  # effectively full test split; module sampling clips to available rows.
QWEN_FEW_SHOT_EXAMPLES_PER_CLASS = 1
QWEN_MAX_EVAL_SAMPLES = None

# Instruction-tuned LLM module variants and decoding search.
INSTRUCTION_LLM_MODEL_KEYS = ["saullm_7b", "qwen_small", "qwen_7b"]
INSTRUCTION_LLM_TUNE_DECODING_ON_VALIDATION = True
INSTRUCTION_LLM_EVALUATE_TEST = True
INSTRUCTION_LLM_MAX_EXAMPLES_PER_SPLIT = None
INSTRUCTION_LLM_BATCH_SIZE = 4
INSTRUCTION_LLM_QUANTIZATION = "none"
INSTRUCTION_LLM_ALLOW_CPU = False
INSTRUCTION_LLM_ERROR_ANALYSIS_EXAMPLES = 50

CUAD_EXTERNAL_MAX_EVAL_SAMPLES = None
CUAD_EXTERNAL_TRANSFORMER_BATCH_SIZE = 16

DOWNLOAD_LEDGAR_IF_MISSING = True
DOWNLOAD_CUAD_IF_MISSING = True
USE_HF_CACHE = True
FORCE_REDOWNLOAD = False

def safe_variant_key(value: str) -> str:
    return re.sub(r"[^a-zA-Z0-9]+", "_", str(value).lower()).strip("_") or "variant"

paths = build_project_paths(PROJECT_ROOT)
DEVICE = seed_everything(SEED)


In [ ]:
# @title
# Exhaustive-run guardrail: fail early if this notebook is not actually set to run the optional stages.
assert (PROJECT_ROOT / "modules").is_dir(), f"Missing modules folder at {PROJECT_ROOT / 'modules'}"

required_true_flags = {
    "RUN_CLASSICAL_MODELS": RUN_CLASSICAL_MODELS,
    "RUN_NAIVE_BAYES": RUN_NAIVE_BAYES,
    "RUN_SEQUENCE_MODEL": RUN_SEQUENCE_MODEL,
    "RUN_CONFIGURED_TRANSFORMER_BASELINES": RUN_CONFIGURED_TRANSFORMER_BASELINES,
    "RUN_TRANSFORMER": RUN_TRANSFORMER,
    "RUN_TRANSFORMER_HPT": RUN_TRANSFORMER_HPT,
    "RUN_QWEN_BASELINE": RUN_QWEN_BASELINE,
    "RUN_INSTRUCTION_LLM_EVAL": RUN_INSTRUCTION_LLM_EVAL,
    "RUN_AGENTIC_EXTENSION": RUN_AGENTIC_EXTENSION,
    "RUN_CUAD_EXTERNAL_EVAL": RUN_CUAD_EXTERNAL_EVAL,
}
for flag_name, flag_value in required_true_flags.items():
    assert flag_value is True, f"{flag_name} is not True"

assert TRANSFORMER_HPT_SMOKE_TEST is False, "TRANSFORMER_HPT_SMOKE_TEST must be False for normal exhaustive mode"
assert HPT_MAX_TRAIN_SAMPLES is None, "HPT_MAX_TRAIN_SAMPLES should be None for full-data HPT"
assert HPT_MAX_VALIDATION_SAMPLES is None, "HPT_MAX_VALIDATION_SAMPLES should be None for full-data HPT"
assert HPT_MAX_EVAL_SAMPLES is None, "HPT_MAX_EVAL_SAMPLES should be None for full-data HPT"
assert CUAD_EXTERNAL_MAX_EVAL_SAMPLES is None, "CUAD_EXTERNAL_MAX_EVAL_SAMPLES should be None for full CUAD external evaluation"

variant_summary = pd.DataFrame([
    {"stage": "classical", "variants": CLASSICAL_TOKENIZER_VARIANTS, "trials_or_models": "LR + SVM + MultinomialNB + ComplementNB via module grid"},
    {"stage": "sequence", "variants": ["BiLSTM"], "trials_or_models": "1 configured neural sequence baseline"},
    {"stage": "configured_transformers", "variants": TRANSFORMER_MODEL_VARIANTS, "trials_or_models": "fixed baseline for each model"},
    {"stage": "transformer_hpt", "variants": TRANSFORMER_HPT_MODEL_VARIANTS, "trials_or_models": f"{HPT_RANDOM_TRIALS} random + {HPT_BAYES_TRIALS} Bayesian per model"},
    {"stage": "qwen_prompting", "variants": QWEN_MODEL_VARIANTS, "trials_or_models": "zero-shot + static few-shot + retrieval few-shot per model"},
    {"stage": "cuad_external", "variants": ["CUAD external eval"], "trials_or_models": "enabled, no sample cap"},
])
display(variant_summary)
print("✅ Normal exhaustive mode confirmed: all notebook-level optional stages are enabled and module folder is present.")


In [ ]:
# @title
# Machine-checkable exhaustive variant manifest.
# This cell exists so the notebook fails early if optional stages are accidentally disabled again.
from pathlib import Path as _Path
import json as _json

EXHAUSTIVE_VARIANT_MANIFEST = {
    "requires_modules_folder": str(PROJECT_ROOT / "modules"),
    "flags": required_true_flags,
    "smoke_mode": TRANSFORMER_HPT_SMOKE_TEST,
    "sample_caps": {
        "HPT_MAX_TRAIN_SAMPLES": HPT_MAX_TRAIN_SAMPLES,
        "HPT_MAX_VALIDATION_SAMPLES": HPT_MAX_VALIDATION_SAMPLES,
        "HPT_MAX_EVAL_SAMPLES": HPT_MAX_EVAL_SAMPLES,
        "QWEN_MAX_EVAL_SAMPLES": QWEN_MAX_EVAL_SAMPLES,
        "INSTRUCTION_LLM_MAX_EXAMPLES_PER_SPLIT": INSTRUCTION_LLM_MAX_EXAMPLES_PER_SPLIT,
        "CUAD_EXTERNAL_MAX_EVAL_SAMPLES": CUAD_EXTERNAL_MAX_EVAL_SAMPLES,
    },
    "classical": {
        "tokenizers": CLASSICAL_TOKENIZER_VARIANTS,
        "models_exposed_by_module": ["logistic_regression", "linear_svm", "multinomial_nb", "complement_nb"],
        "max_features_list": MAX_FEATURES_LIST,
        "ngram_ranges": [list(x) for x in NGRAM_RANGES],
        "min_df_list": MIN_DF_LIST,
        "c_values": CLASSICAL_C_VALUES,
        "nb_alpha_values": CLASSICAL_NB_ALPHA_VALUES,
    },
    "sequence": {"enabled": RUN_SEQUENCE_MODEL, "variant": "BiLSTM"},
    "configured_transformers": TRANSFORMER_MODEL_VARIANTS,
    "transformer_hpt": {
        "models": TRANSFORMER_HPT_MODEL_VARIANTS,
        "random_trials_per_model": HPT_RANDOM_TRIALS,
        "bayes_trials_per_model": HPT_BAYES_TRIALS,
        "final_retrain_epochs": HPT_FINAL_RETRAIN_EPOCHS,
    },
    "qwen_prompting": {
        "models": QWEN_MODEL_VARIANTS,
        "prompt_modes_inside_module": ["zero_shot", "static_few_shot", "retrieval_few_shot"],
    },
    "instruction_llms": {
        "model_keys": INSTRUCTION_LLM_MODEL_KEYS,
        "tune_decoding_on_validation": INSTRUCTION_LLM_TUNE_DECODING_ON_VALIDATION,
        "evaluate_test": INSTRUCTION_LLM_EVALUATE_TEST,
        "quantization": INSTRUCTION_LLM_QUANTIZATION,
    },
    "agentic_review": {"enabled": RUN_AGENTIC_EXTENSION},
    "cuad_external": {"enabled": RUN_CUAD_EXTERNAL_EVAL, "max_eval_samples": CUAD_EXTERNAL_MAX_EVAL_SAMPLES},
    "wandb": {
        "enabled": RUN_WANDB,
        "mode": WANDB_MODE,
        "run_strategy": WANDB_RUN_STRATEGY,
        "enabled_stages": WANDB_ENABLED_STAGES,
        "stage_group": WANDB_STAGE_GROUP,
    },
}

assert all(EXHAUSTIVE_VARIANT_MANIFEST["flags"].values()), EXHAUSTIVE_VARIANT_MANIFEST["flags"]
assert EXHAUSTIVE_VARIANT_MANIFEST["smoke_mode"] is False
assert all(value is None for value in EXHAUSTIVE_VARIANT_MANIFEST["sample_caps"].values()), EXHAUSTIVE_VARIANT_MANIFEST["sample_caps"]

manifest_path = paths.project_root / "outputs" / "exhaustive_variant_manifest.json"
manifest_path.parent.mkdir(parents=True, exist_ok=True)
manifest_path.write_text(_json.dumps(EXHAUSTIVE_VARIANT_MANIFEST, indent=2), encoding="utf-8")
print(f"✅ Wrote exhaustive variant manifest: {manifest_path}")
display(pd.DataFrame([
    {"stage": "classical", "variants": str(EXHAUSTIVE_VARIANT_MANIFEST["classical"]["tokenizers"]), "enabled": RUN_CLASSICAL_MODELS},
    {"stage": "sequence", "variants": "BiLSTM", "enabled": RUN_SEQUENCE_MODEL},
    {"stage": "configured_transformers", "variants": str(TRANSFORMER_MODEL_VARIANTS), "enabled": RUN_CONFIGURED_TRANSFORMER_BASELINES},
    {"stage": "transformer_hpt", "variants": str(TRANSFORMER_HPT_MODEL_VARIANTS), "enabled": RUN_TRANSFORMER_HPT},
    {"stage": "qwen_prompting", "variants": str(QWEN_MODEL_VARIANTS), "enabled": RUN_QWEN_BASELINE},
    {"stage": "instruction_llms", "variants": str(INSTRUCTION_LLM_MODEL_KEYS), "enabled": RUN_INSTRUCTION_LLM_EVAL},
    {"stage": "agentic_review", "variants": "agentic_review", "enabled": RUN_AGENTIC_EXTENSION},
    {"stage": "cuad_external", "variants": "CUAD", "enabled": RUN_CUAD_EXTERNAL_EVAL},
]))


In [ ]:
# @title
# Static sanity check for the exhaustive notebook wiring.
# This checks the notebook-level settings before expensive jobs start.
assert not hasattr(paths, "outputs_dir"), "ProjectPaths has no outputs_dir; use paths.project_root / 'outputs' instead."
assert (paths.project_root / "outputs").exists() or True
assert RUN_CLASSICAL_MODELS and RUN_SEQUENCE_MODEL and RUN_CONFIGURED_TRANSFORMER_BASELINES
assert RUN_TRANSFORMER and RUN_TRANSFORMER_HPT and RUN_QWEN_BASELINE
assert RUN_INSTRUCTION_LLM_EVAL and RUN_AGENTIC_EXTENSION and RUN_CUAD_EXTERNAL_EVAL
print("✅ Notebook wiring sanity check passed: exhaustive flags are enabled and output paths use the real ProjectPaths API.")


Weights and Biases Setup

In [ ]:
# @title
print(f"Project root: {paths.project_root}")
print(f"Colab runtime: {IN_COLAB}")
print(f"Raw LEDGAR directory: {paths.ledgar_raw_dir}")
print(f"Results directory: {paths.results_dir}")
print(f"Device: {DEVICE}")

In [ ]:
# @title
try:
    import torch

    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU count: {torch.cuda.device_count()}")
        for gpu_idx in range(torch.cuda.device_count()):
            print(f"GPU {gpu_idx}: {torch.cuda.get_device_name(gpu_idx)}")
    else:
        print("GPU is unavailable. Heavy Transformer/Qwen/LLM sections will skip or fail gracefully.")
except Exception:
    print("PyTorch is unavailable. Heavy model sections will skip if they require it.")

import os
os.environ.setdefault("WANDB_CONSOLE", "off")
os.environ.setdefault("WANDB_DISABLE_CODE", "true")
os.environ.setdefault("WANDB_START_METHOD", "thread")

wandb_run = None
WANDB_ACTIVE = False
WANDB_CURRENT_STAGE = None


def _wandb_stage_set() -> set[str] | str:
    raw = str(globals().get("WANDB_ENABLED_STAGES", "all")).strip()
    if not raw or raw.lower() == "all":
        return "all"
    return {item.strip() for item in raw.split(",") if item.strip()}


def _wandb_stage_allowed(stage_name: str) -> bool:
    if not globals().get("RUN_WANDB", False):
        return False
    strategy = str(globals().get("WANDB_RUN_STRATEGY", "per_stage")).strip().lower()
    if strategy in {"disabled", "disable", "off", "none", "false", "0"}:
        return False
    enabled = _wandb_stage_set()
    return enabled == "all" or stage_name in enabled


def _base_wandb_config(stage_name: str, extra_config: dict | None = None) -> dict:
    config = {
        "stage": stage_name,
        "seed": SEED,
        "dataset_name": DATASET_NAME,
        "top_k_labels": TOP_K_LABELS,
        "run_classical_models": RUN_CLASSICAL_MODELS,
        "classical_tokenizer_variants": CLASSICAL_TOKENIZER_VARIANTS,
        "run_sequence_model": RUN_SEQUENCE_MODEL,
        "run_configured_transformer_baselines": RUN_CONFIGURED_TRANSFORMER_BASELINES,
        "run_transformer": RUN_TRANSFORMER,
        "run_transformer_hpt": RUN_TRANSFORMER_HPT,
        "transformer_model_variants": TRANSFORMER_MODEL_VARIANTS,
        "transformer_hpt_model_variants": TRANSFORMER_HPT_MODEL_VARIANTS,
        "hpt_random_trials": HPT_RANDOM_TRIALS,
        "hpt_bayes_trials": HPT_BAYES_TRIALS,
        "hpt_final_retrain_epochs": HPT_FINAL_RETRAIN_EPOCHS,
        "run_qwen_baseline": RUN_QWEN_BASELINE,
        "qwen_model_variants": QWEN_MODEL_VARIANTS,
        "run_instruction_llm_eval": RUN_INSTRUCTION_LLM_EVAL,
        "instruction_llm_model_keys": INSTRUCTION_LLM_MODEL_KEYS,
        "run_agentic_extension": RUN_AGENTIC_EXTENSION,
        "run_cuad_external_eval": RUN_CUAD_EXTERNAL_EVAL,
        "run_naive_bayes": RUN_NAIVE_BAYES,
        "device": str(DEVICE),
        "wandb_run_strategy": WANDB_RUN_STRATEGY,
        "wandb_enabled_stages": WANDB_ENABLED_STAGES,
        "log_text_tables": WANDB_LOG_TEXT_TABLES,
        "log_model_files": WANDB_LOG_MODEL_FILES,
    }
    if extra_config:
        config.update(extra_config)
    return config


def start_wandb_stage(stage_name: str, config: dict | None = None, tags: list[str] | None = None):
    """Start W&B according to WANDB_RUN_STRATEGY.

    - per_stage: finish any previous run, then start a fresh run for this stage.
    - single: start one run once and reuse it across the notebook.
    - disabled: no-op.
    """
    global wandb_run, WANDB_ACTIVE, WANDB_CURRENT_STAGE

    strategy = str(globals().get("WANDB_RUN_STRATEGY", "per_stage")).strip().lower()
    if not _wandb_stage_allowed(stage_name):
        print(f"W&B not started for {stage_name!r} (strategy={strategy}, RUN_WANDB={RUN_WANDB}).")
        WANDB_ACTIVE = False
        WANDB_CURRENT_STAGE = None
        return None

    try:
        import wandb

        if strategy == "single":
            if wandb.run is not None:
                wandb_run = wandb.run
                WANDB_ACTIVE = True
                WANDB_CURRENT_STAGE = stage_name
                print(f"Reusing existing single W&B run for stage {stage_name}: {wandb_run.name}")
                try:
                    wandb.config.update({f"stage_seen/{stage_name}": True}, allow_val_change=True)
                except Exception:
                    pass
                return wandb_run

            run_name = globals().get("WANDB_SINGLE_RUN_NAME", "ledgar-all-variants-single-notebook")
        else:
            if wandb.run is not None:
                print(f"Finishing previous W&B run before starting stage {stage_name}.")
                wandb.finish(exit_code=0)
            run_name = stage_name

        wandb_run = wandb.init(
            project=WANDB_PROJECT,
            entity=WANDB_ENTITY,
            mode=WANDB_MODE,
            name=run_name,
            group=WANDB_STAGE_GROUP,
            job_type=stage_name,
            tags=tags or ["ledgar", "coursework", "all-variants", "single-notebook", stage_name],
            config=_base_wandb_config(stage_name, config),
            reinit=True,
            settings=wandb.Settings(start_method="thread"),
        )
        WANDB_ACTIVE = True
        WANDB_CURRENT_STAGE = stage_name
        print(f"W&B active for stage {stage_name}: {wandb_run.name}")
        return wandb_run

    except Exception as exc:
        print(f"W&B could not start for {stage_name}: {type(exc).__name__}: {exc}")
        print("Continuing without W&B for this stage.")
        wandb_run = None
        WANDB_ACTIVE = False
        WANDB_CURRENT_STAGE = None
        return None


def finish_wandb_stage(final: bool = False):
    """Finish W&B cleanly for per-stage mode, or only at final cleanup for single mode."""
    global wandb_run, WANDB_ACTIVE, WANDB_CURRENT_STAGE

    strategy = str(globals().get("WANDB_RUN_STRATEGY", "per_stage")).strip().lower()
    should_finish = strategy == "per_stage" or final

    try:
        import wandb
        if should_finish and wandb.run is not None:
            wandb.finish(exit_code=0)
            print("W&B run finished cleanly.")
            wandb_run = None
            WANDB_ACTIVE = False
            WANDB_CURRENT_STAGE = None
        elif wandb.run is not None:
            print(f"Keeping single W&B run open after stage {WANDB_CURRENT_STAGE}.")
    except Exception as exc:
        print(f"W&B finish warning: {type(exc).__name__}: {exc}")

    try:
        import gc
        gc.collect()
    except Exception:
        pass

    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass

print("W&B strategy:", WANDB_RUN_STRATEGY)
print("W&B enabled stages:", WANDB_ENABLED_STAGES)
print("Set WANDB_RUN_STRATEGY to 'per_stage', 'single', or 'disabled'.")


In [ ]:
# @title
# W&B authentication via Colab Secret only
try:
    from google.colab import userdata

    wandb_key = userdata.get("WANDB_API_KEY")
    if wandb_key:
        os.environ["WANDB_API_KEY"] = wandb_key
        print("WANDB_API_KEY loaded from Colab Secrets.")
    else:
        print("WANDB_API_KEY not found in Colab Secrets.")
except Exception as exc:
    print(f"Colab Secrets unavailable: {type(exc).__name__}: {exc}")
start_wandb_stage("01_setup_config", {
    "stage_type": "setup_config_imports",
    "note": "Setup/config imports and exhaustive flags were initialised before this W&B helper cell; this run records the completed setup stage.",
})
finish_wandb_stage()  # end 01_setup_config


## 2. Dataset Download and Raw Setup

This stage obtains the raw datasets without training or preprocessing models. LEDGAR remains the main classification dataset. CUAD is treated separately because it is structured as a contract-review question-answering/span-extraction dataset rather than a direct clause-classification dataset.

Data governance choices:

- LEDGAR is loaded from Hugging Face with `load_dataset("coastalcph/lex_glue", "ledgar")` when local JSONL files are missing.
- Official LEDGAR train, validation, and test splits are preserved when available.
- Raw LEDGAR split exports are saved under `data/raw/lexglue_ledgar/` as JSONL files.
- CUAD raw files are downloaded from `theatticusproject/cuad` when available, but CUAD is not merged with LEDGAR.
- If CUAD is missing, the notebook prints a clear message and continues with LEDGAR.

Inputs and outputs:

| Input | Output |
|---|---|
| Hugging Face LEDGAR or local JSONL | `ledgar_raw_splits` dictionary with train/validation/test DataFrames |
| Optional CUAD raw files | `cuad_clause_df` containing extracted span-level examples for optional inspection |

This stage deliberately does not select labels, encode classes, train models, or compute metrics.


CUAD Dataset

In [ ]:
# @title
start_wandb_stage("02_raw_dataset_setup", {"stage_type": "dataset_download_raw_setup"})


In [ ]:
# @title
ledgar_raw_splits = load_or_download_ledgar(
    paths,
    download_if_missing=DOWNLOAD_LEDGAR_IF_MISSING,
    force_redownload=FORCE_REDOWNLOAD,
)

cuad_json_path, master_clauses_path = download_cuad_if_missing(
    paths,
    download_if_missing=DOWNLOAD_CUAD_IF_MISSING,
    force_redownload=FORCE_REDOWNLOAD,
)

raw_cuad_json, master_clauses_df = load_cuad_raw_files(cuad_json_path, master_clauses_path)
cuad_clause_df = adapt_cuad_to_clause_classification(raw_cuad_json)
print_dataset_availability(ledgar_raw_splits, cuad_json_path, master_clauses_path, cuad_clause_df)

In [ ]:
# @title
finish_wandb_stage()  # end 02_raw_dataset_setup


## 3. LEDGAR Preprocessing and EDA

This stage is intentionally split into small cells so each lecture/lab technique can be inspected before the full dataset is processed.

Preprocessing changes the raw text into a cleaner form. Feature extraction converts text into numeric vectors. Modelling starts only after those vectors are produced.


### Cell 1 - Separate preprocessing, feature extraction, and modelling

| Stage | What happens here | Examples in this notebook |
|---|---|---|
| Preprocessing | Clean or tokenise raw clause text | HTML/entity cleanup, whitespace normalisation, regex tokens, negation-aware tokens, BPE inspection |
| Feature extraction | Convert text/tokens into numeric vectors | BoW, TF-IDF, unigrams, bigrams |
| Modelling | Fit a classifier using features and labels | Logistic Regression, Linear SVM, Naive Bayes |

Logistic Regression is therefore not preprocessing; it appears later as a model.


In [ ]:
# @title
import importlib
import modules.preprocessing as preprocessing

importlib.reload(preprocessing)
print("Reloaded modules.preprocessing")

In [ ]:
# @title
start_wandb_stage("03_preprocessing_eda", {"stage_type": "preprocessing_eda"})


In [ ]:
# @title
from modules.preprocessing import (
    bpe_encode_text,
    bpe_encode_word,
    clean_html_entities,
    corpus_word_frequencies,
    create_ledgar_eda,
    legal_safe_tokenise,
    negation_aware_tokenise,
    preprocess_ledgar,
    preprocessing_technique_rundown,
    regex_tokenise,
    train_bpe_tokeniser,
    write_preprocessing_rundown,
)


In [ ]:
# @title
#  Show one raw contract clause example before preprocessing.
if ledgar_raw_splits:
    raw_train_df = ledgar_raw_splits["train"]
    raw_text_column = next(column for column in ("text", "provision", "clause", "contract_text") if column in raw_train_df.columns)
    raw_clause = str(raw_train_df.iloc[0][raw_text_column])
else:
    raw_text_column = "text"
    raw_clause = "The Borrower shall not be liable for any indirect damages &amp; shall give notice under Section 5.1."

print(raw_clause[:1000])


In [ ]:
# @title
html_clean_clause = clean_html_entities(raw_clause)
# Apply whitespace normalisation.
whitespace_clean_clause = normalise_whitespace(html_clean_clause)
print(whitespace_clean_clause[:1000])

In [ ]:
# @title
# Run regex tokenisation.
regex_tokens = regex_tokenise(whitespace_clean_clause)
print(regex_tokens[:80])
print(f"Token count: {len(regex_tokens)}")


In [ ]:
# @title
# Run legal-safe lowercased tokenisation.
# This lowercases tokens for feature extraction without overwriting the stored clause text.
legal_tokens = legal_safe_tokenise(whitespace_clean_clause)
print(legal_tokens[:80])


In [ ]:
# @title
# Run Week 2 negation-aware tokenisation.
negation_tokens = negation_aware_tokenise(whitespace_clean_clause)
print(negation_tokens[:100])


In [ ]:
# @title
# Show why stopword removal is skipped for legal clauses.
# These words are often treated as stopwords in generic NLP, but they can change legal meaning.
legal_stopword_examples = {"no", "not", "shall", "may", "unless", "except", "without"}
kept_legal_tokens = [token for token in legal_tokens if token in legal_stopword_examples]

print("Legal stopword-like tokens kept:", kept_legal_tokens)
print("Default decision: do not remove stopwords for contract clause classification.")


In [ ]:
# @title
# Train a small Week 2/3 BPE tokenizer on training clause samples.
if ledgar_raw_splits:
    bpe_training_texts = ledgar_raw_splits["train"][raw_text_column].astype(str).head(250).tolist()
else:
    bpe_training_texts = [whitespace_clean_clause]

bpe_word_counts = corpus_word_frequencies(bpe_training_texts, max_words=2000)
bpe_merges, bpe_vocab = train_bpe_tokeniser(bpe_word_counts, num_merges=50)

print(f"BPE training words: {len(bpe_word_counts)}")
print(f"BPE merges learned: {len(bpe_merges)}")
print(list(bpe_merges.items())[:10])


In [ ]:
# @title
# Run BPE encoding/OOV examples.
for word in ["lowest", "lover", "newly", "unwanted", "indemnification", "xyz"]:
    print(f"{word:20} -> {bpe_encode_word(word, bpe_merges)}")

print("Clause BPE preview:")
print(bpe_encode_text(whitespace_clean_clause, bpe_merges)[:100])


In [ ]:
# @title
# Preprocess full LEDGAR splits and save processed outputs.
processed_splits, label2id, id2label = preprocess_ledgar(
    ledgar_raw_splits,
    paths,
    top_k_labels=TOP_K_LABELS,
    dataset_name=DATASET_NAME,
)

split_summary = create_ledgar_eda(processed_splits, paths.results_dir)

if processed_splits:
    train_df = processed_splits["train"]
    validation_df = processed_splits["validation"]
    test_df = processed_splits["test"]
    label_names = [id2label[i] for i in sorted(id2label)]
    display(split_summary)
    display(pd.DataFrame({"label": label_names}))
else:
    train_df = validation_df = test_df = pd.DataFrame(
        columns=["text", "label", "label_id", "split", "source_dataset"]
    )
    label_names = []
    print("Main LEDGAR experiment cannot run without LEDGAR data.")


In [ ]:
# @title

print("Processed split summary:")
for split, df in processed_splits.items():
    print(
        split,
        "rows:", len(df),
        "classes:", df["label"].nunique(),
        "columns:", list(df.columns),
    )

print("\nTraining label distribution:")
display(processed_splits["train"]["label"].value_counts().head(20))

dataset_summary_path = paths.processed_data_dir / "dataset_summary.json"
if dataset_summary_path.exists():
    print("\nDataset summary:")
    display(json.loads(dataset_summary_path.read_text(encoding="utf-8")))

leakage_audit_path = paths.project_root / "outputs" / "leakage_audit.json"
if leakage_audit_path.exists():
    leakage_audit = json.loads(leakage_audit_path.read_text(encoding="utf-8"))

    print("\nDuplicate rows removed by split:")
    display(leakage_audit.get("duplicate_rows_removed_by_split", {}))

    print("\nCross-split overlaps before deduplication:")
    display(leakage_audit.get("cross_split_overlaps_before_deduplication", {}))

    print("\nCross-split overlaps after deduplication:")
    display(leakage_audit.get("cross_split_overlaps_after_deduplication", {}))

In [ ]:
# @title
# Write a readable rundown of preprocessing and feature techniques.
rundown_path = write_preprocessing_rundown(paths.project_root / "outputs" / "preprocessing_techniques.md")
print(f"Wrote: {rundown_path}")
print(preprocessing_technique_rundown())


In [ ]:
finish_wandb_stage()  # end 03_preprocessing_eda


## 4. Shared Result State

This short stage creates shared containers used by the later model sections.

- `completed_results` stores one row per completed or skipped model run.
- `prediction_tables` stores per-example predictions for error analysis.
- `trained_models` stores reusable fitted model objects when available.

Governance purpose: every model section appends to the same result structure, so the final comparison table is generated from actual run outputs rather than manually entered values.


Variable Initialization

In [ ]:
start_wandb_stage("04_shared_result_state", {"stage_type": "shared_result_state"})


In [ ]:
completed_results = []

prediction_tables = {}

trained_models = {}

In [ ]:
finish_wandb_stage()  # end 04_shared_result_state


## 5. Dummy Baselines

This stage evaluates non-learning baselines. These baselines are important because they establish a minimum reference point before interpreting more complex models.

Baselines used:

| Model | Behavior | Why it matters |
|---|---|---|
| `random_uniform` | Samples uniformly from the selected label IDs. | Tests performance expected from chance under equal class probability. |
| `random_train_distribution` | Samples labels according to the training label distribution. | Reflects class imbalance without learning from text. |
| `majority_baseline` | Always predicts the most frequent training label. | Provides a strong imbalance-aware dummy baseline for accuracy comparison. |

Evaluation metrics saved for each baseline:

- accuracy
- macro-F1
- weighted-F1
- per-class precision/recall/F1 via classification report
- confusion matrix

Macro-F1 is the primary governance metric because it penalises poor performance on minority classes more clearly than accuracy.


In [ ]:
start_wandb_stage("05_dummy_baselines", {"stage_type": "dummy_baselines"})


In [ ]:
baseline_results, baseline_prediction_tables = run_baseline_experiments(

    train_df,

    test_df,

    id2label,

    paths.results_dir,

    dataset_name=DATASET_NAME,

    seed=SEED,
)


completed_results.extend(baseline_results)

prediction_tables.update(baseline_prediction_tables)

if baseline_results:
    display(pd.DataFrame(baseline_results)[["model_name", "accuracy", "macro_f1", "weighted_f1", "notes"]])

In [ ]:
finish_wandb_stage()  # end 05_dummy_baselines


## 6. Classical TF-IDF Models

This stage trains sparse-text supervised models using TF-IDF features. These models are fast, interpretable at the feature level, and provide strong non-neural baselines for legal text classification.


Selection protocol:

1. Train each configuration on the LEDGAR training split.
2. Select the best configuration using validation macro-F1.
3. Evaluate selected models on the test split once.
4. Save the best classical pipeline and vectorizer artifacts separately.

Explainability note: TF-IDF models are useful for coursework governance because their decisions are linked to sparse lexical features rather than hidden contextual embeddings.


Variable Initialization

In [ ]:
# Initialize variables to track the best classical model and its name, which will be updated after running classical experiments.

best_classical_model = None

best_classical_name = None

Training for Classical Machine-Learning Models

Feature extraction hyperparameters:

| Hyperparameter | Values |
|---|---|
| `tokenizer` | `negation_aware` lab-grounded tokenizer |
| `max_features` | `10000`, `30000` |
| `ngram_range` | unigram `(1, 1)`, unigram+bigram `(1, 2)` |
| `lowercase` | handled inside the tokenizer |
| `stop_words` | `None` because legal stopword-like terms can change clause meaning |

The classifier is trained only after these features are built.


### Feature extraction

The cells below inspect Bag-of-Words and TF-IDF features before any classifier is trained. This keeps the Week 3 lab feature work separate from Logistic Regression, Linear SVM, and Naive Bayes.


In [ ]:
# Build Bag-of-Words features with CountVectorizer.
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

feature_sample_texts = train_df["text"].astype(str).head(200).tolist() if not train_df.empty else [
    "Borrower shall not be liable for indirect damages.",
    "Either party may terminate this Agreement on notice.",
]

bow_vectorizer = CountVectorizer(
    tokenizer=legal_safe_tokenise,
    token_pattern=None,
    lowercase=False,
)
bow_features = bow_vectorizer.fit_transform(feature_sample_texts)

print(f"BoW matrix shape: {bow_features.shape}")
print(f"BoW vocabulary size: {len(bow_vectorizer.vocabulary_)}")


In [ ]:
# Cell 3 - Inspect non-zero BoW features for one clause.
feature_names = bow_vectorizer.get_feature_names_out()
first_bow = bow_features[0].tocoo()
first_bow_df = pd.DataFrame({
    "feature": feature_names[first_bow.col],
    "count": first_bow.data,
}).sort_values(["count", "feature"], ascending=[False, True])

print(feature_sample_texts[0])
display(first_bow_df.head(30))


In [ ]:
# Build TF-IDF unigram features.
tfidf_unigram_vectorizer = TfidfVectorizer(
    tokenizer=negation_aware_tokenise,
    token_pattern=None,
    lowercase=False,
    ngram_range=(1, 1),
    max_features=MAX_FEATURES_LIST[0],
)
tfidf_unigram_features = tfidf_unigram_vectorizer.fit_transform(feature_sample_texts)

print(f"TF-IDF unigram matrix shape: {tfidf_unigram_features.shape}")
print(tfidf_unigram_vectorizer.get_feature_names_out()[:40])


In [ ]:
# Build TF-IDF unigram+bigram features.
tfidf_unibigram_vectorizer = TfidfVectorizer(
    tokenizer=negation_aware_tokenise,
    token_pattern=None,
    lowercase=False,
    ngram_range=(1, 2),
    max_features=MAX_FEATURES_LIST[0],
)
tfidf_unibigram_features = tfidf_unibigram_vectorizer.fit_transform(feature_sample_texts)

print(f"TF-IDF unigram+bigram matrix shape: {tfidf_unibigram_features.shape}")
print(tfidf_unibigram_vectorizer.get_feature_names_out()[:40])


In [ ]:
# Compare feature matrix shapes before modelling.
feature_shape_summary = pd.DataFrame([
    {"representation": "BoW unigrams", "rows": bow_features.shape[0], "features": bow_features.shape[1]},
    {"representation": "TF-IDF unigrams", "rows": tfidf_unigram_features.shape[0], "features": tfidf_unigram_features.shape[1]},
    {"representation": "TF-IDF unigrams+bigrams", "rows": tfidf_unibigram_features.shape[0], "features": tfidf_unibigram_features.shape[1]},
])
display(feature_shape_summary)


In [ ]:
import importlib
import modules.preprocessing as preprocessing
import modules.classical_models as classical_models

importlib.reload(preprocessing)
importlib.reload(classical_models)

from modules.preprocessing import (
    legal_safe_tokenise,
    negation_aware_tokenise,
)

from modules.classical_models import run_classical_experiments

print("Reloaded preprocessing and classical_models")

### Statistical Modelling Hyperparameter Tning

The next cells train Logistic Regression, Linear SVM, and Naive Bayes using the TF-IDF feature setup. These classifiers are modelling steps, not preprocessing.


In [ ]:
# @title
start_wandb_stage("06_classical_tfidf_all_variants", {
    "stage_type": "classical_tfidf_all_variants",
    "tokenizer_variants": CLASSICAL_TOKENIZER_VARIANTS,
    "max_features_list": MAX_FEATURES_LIST,
    "ngram_ranges": NGRAM_RANGES,
    "min_df_list": MIN_DF_LIST,
    "c_values": CLASSICAL_C_VALUES,
    "nb_alpha_values": CLASSICAL_NB_ALPHA_VALUES,
})

try:
    # Training with classical variants.
    # This runs all classical model families exposed by modules.classical_models:
    # Logistic Regression, Linear SVM, Multinomial NB, Complement NB,
    # across TF-IDF max_features, ngram_range, min_df, C/alpha, class_weight, and tokenizer variants.

    classical_output = {
        "results": [],
        "prediction_tables": {},
        "best_model": None,
        "best_model_name": None,
        "validation_grid": pd.DataFrame(),
    }
    best_classical_model = None
    best_classical_name = None
    best_classical_macro_f1 = -1.0
    classical_validation_frames = []

    if not RUN_CLASSICAL_MODELS:
        print("RUN_CLASSICAL_MODELS is False; skipping classical variants.")
    else:
        for tokenizer_name in CLASSICAL_TOKENIZER_VARIANTS:
            print(f"\n=== Classical variant tokenizer={tokenizer_name} ===")
            variant_output = run_classical_experiments(
                train_df,
                validation_df,
                test_df,
                id2label,
                paths.results_dir,
                max_features_list=MAX_FEATURES_LIST,
                ngram_ranges=NGRAM_RANGES,
                min_df_list=MIN_DF_LIST,
                c_values=CLASSICAL_C_VALUES,
                nb_alpha_values=CLASSICAL_NB_ALPHA_VALUES,
                dataset_name=DATASET_NAME,
                seed=SEED,
                run_naive_bayes=RUN_NAIVE_BAYES,
                tokenizer_name=tokenizer_name,
            )

            # Preserve the module's standard output directory before the next tokenizer overwrites it.
            standard_dir = paths.results_dir / "classical"
            archive_dir = paths.results_dir / f"classical_{safe_variant_key(tokenizer_name)}"
            if standard_dir.exists():
                if archive_dir.exists():
                    shutil.rmtree(archive_dir)
                shutil.copytree(standard_dir, archive_dir)

            variant_results = []
            for result in variant_output.get("results", []):
                result_copy = dict(result)
                original_name = result_copy.get("model_name", "classical")
                result_copy["model_name"] = f"{original_name}_{tokenizer_name}"
                result_copy["notes"] = f"Tokenizer={tokenizer_name}. " + str(result_copy.get("notes", ""))
                result_copy["classical_tokenizer"] = tokenizer_name
                result_copy["evidence_archive_dir"] = str(archive_dir)
                variant_results.append(result_copy)
                completed_results.append(result_copy)

                if pd.notna(result_copy.get("macro_f1")) and float(result_copy["macro_f1"]) > best_classical_macro_f1:
                    best_classical_macro_f1 = float(result_copy["macro_f1"])
                    best_classical_model = variant_output.get("best_model")
                    best_classical_name = result_copy["model_name"]

            for model_name, pred_df in variant_output.get("prediction_tables", {}).items():
                pred_copy = pred_df.copy()
                pred_copy["model_name"] = f"{model_name}_{tokenizer_name}"
                prediction_tables[f"{model_name}_{tokenizer_name}"] = pred_copy
                classical_output["prediction_tables"][f"{model_name}_{tokenizer_name}"] = pred_copy

            validation_grid = variant_output.get("validation_grid", pd.DataFrame()).copy()
            if not validation_grid.empty:
                validation_grid["tokenizer"] = tokenizer_name
                classical_validation_frames.append(validation_grid)

            classical_output["results"].extend(variant_results)

        classical_output["best_model"] = best_classical_model
        classical_output["best_model_name"] = best_classical_name
        classical_output["validation_grid"] = (
            pd.concat(classical_validation_frames, ignore_index=True)
            if classical_validation_frames
            else pd.DataFrame()
        )

        # Re-save combined all-tokenizer evidence into the standard paths used by report/export/audit modules.
        combined_classical_dir = paths.results_dir / "classical"
        combined_classical_dir.mkdir(parents=True, exist_ok=True)
        pd.DataFrame(classical_output["results"]).to_csv(combined_classical_dir / "classical_results.csv", index=False)
        classical_output["validation_grid"].to_csv(combined_classical_dir / "classical_validation_grid.csv", index=False)

        if best_classical_model is not None:
            trained_models["best_classical"] = best_classical_model

        if classical_output["results"]:
            display(pd.DataFrame(classical_output["results"])[["model_name", "accuracy", "macro_f1", "weighted_f1", "notes"]])

        if not classical_output["validation_grid"].empty:
            print(f"Classical validation grid rows: {len(classical_output['validation_grid'])}")

    # W&B logging for classical stage outputs
    if WANDB_ACTIVE and wandb_run is not None and classical_output["results"]:
        try:
            import wandb

            classical_results_df = pd.DataFrame(classical_output["results"])

            for _, row in classical_results_df.iterrows():
                model_key = str(row["model_name"]).replace("/", "_").replace(" ", "_")
                wandb.log({
                    f"classical/{model_key}/accuracy": float(row["accuracy"]),
                    f"classical/{model_key}/macro_f1": float(row["macro_f1"]),
                    f"classical/{model_key}/weighted_f1": float(row["weighted_f1"]),
                })

            wandb.log({
                "classical/results_table": wandb.Table(dataframe=classical_results_df)
            })

            if not classical_output["validation_grid"].empty:
                validation_grid_preview = classical_output["validation_grid"].head(5000).copy()
                wandb.log({
                    "classical/validation_grid_preview": wandb.Table(dataframe=validation_grid_preview)
                })

            print("✅ Logged classical metrics/tables to W&B.")

        except Exception as exc:
            print(f"W&B classical logging skipped: {type(exc).__name__}: {exc}")

finally:
    finish_wandb_stage()  # end 06_classical_tfidf_all_variants

In [ ]:
print(f"Type of negation_aware_tokenise: {type(negation_aware_tokenise)}")
print(f"Location of negation_aware_tokenise: {negation_aware_tokenise.__module__}.{negation_aware_tokenise.__name__}")

## 7. Neural Sequence Baseline

This enabled stage trains a compact BiLSTM classifier on the processed LEDGAR splits. It uses the training split for vocabulary/model fitting, the validation split for best-checkpoint selection by macro-F1, and the test split only after selection.

In this normal exhaustive exhaustive notebook, `RUN_SEQUENCE_MODEL=True`, so this stage is expected to run. Use the normal notebook or set the flag to `False` only for a reduced run.


In [ ]:
start_wandb_stage("07_bilstm", {"stage_type": "sequence_model", "run_sequence_model": RUN_SEQUENCE_MODEL})


In [ ]:
# @title
sequence_output = {"result": None, "predictions": pd.DataFrame(), "history": pd.DataFrame(), "skip_result": None}

if RUN_SEQUENCE_MODEL:
    sequence_output = train_sequence_classifier(
        train_df,
        validation_df,
        test_df,
        id2label,
        paths.results_dir,
        dataset_name=DATASET_NAME,
        config=SequenceModelConfig(seed=SEED, epochs=8, patience=2),
        run_sequence_model=True,
    )

    if sequence_output["result"] is not None:
        completed_results.append(sequence_output["result"])
        prediction_tables["bilstm"] = sequence_output["predictions"]
        display(pd.DataFrame([sequence_output["result"]])[["model_name", "accuracy", "macro_f1", "weighted_f1"]])
    elif sequence_output.get("skip_result") is not None:
        completed_results.append(sequence_output["skip_result"])
else:
    print("BiLSTM sequence baseline skipped because RUN_SEQUENCE_MODEL=False.")


In [ ]:
# @title
finish_wandb_stage()  # end 07_bilstm


## 8. Fine-Tuned Transformer Classifier

This stage optionally fine-tunes a Hugging Face sequence-classification transformer on LEDGAR. The default model is `distilbert-base-uncased` because it is smaller and more practical for coursework hardware than full BERT-size alternatives.

Training settings used by the module:

| Setting | Value |
|---|---:|
| Model | `distilbert-base-uncased` |
| Maximum sequence length | `256` tokens |
| Learning rate | `2e-5` |
| Epochs | `3` |
| Weight decay | `0.01` |
| Batch size | `16` on larger GPUs, otherwise `8` |
| Mixed precision | `fp16=True` when CUDA is available |
| Model selection | best validation `macro_f1` |
| Early stopping | patience `1` when the callback is available |

Runtime governance:

- This section skips gracefully if CUDA/GPU is unavailable.
- Memory or environment failures are caught and recorded as skipped results.
- The transformer is evaluated on the same LEDGAR test labels as the classical models.

Explainability limitation: transformer representations are contextual but less directly inspectable than TF-IDF features, so confusion matrices and misclassified examples are important for interpreting behavior.


In [ ]:
# @title
start_wandb_stage("08_transformers_configured_and_hpt_all_variants", {
    "stage_type": "transformers_configured_and_hpt_all_variants",
    "run_configured_transformer_baselines": RUN_CONFIGURED_TRANSFORMER_BASELINES,
    "run_transformer_hpt": RUN_TRANSFORMER_HPT,
    "transformer_model_variants": TRANSFORMER_MODEL_VARIANTS,
    "transformer_hpt_model_variants": TRANSFORMER_HPT_MODEL_VARIANTS,
    "random_trials": HPT_RANDOM_TRIALS,
    "bayes_trials": HPT_BAYES_TRIALS,
})


In [ ]:
# @title
transformer_outputs = []
transformer_hpt_outputs = {}
hpt_output = None
transformer_output = {"result": None, "predictions": pd.DataFrame(), "trainer": None, "skip_result": None}

# 1) Run fixed/configured transformer baselines for every transformer variant.
if RUN_CONFIGURED_TRANSFORMER_BASELINES:
    for model_name in TRANSFORMER_MODEL_VARIANTS:
        print(f"\n=== Configured transformer baseline: {model_name} ===")
        output = train_transformer_classifier(
            train_df,
            validation_df,
            test_df,
            id2label,
            paths.results_dir,
            model_name=model_name,
            max_length=MAX_TRANSFORMER_LENGTH,
            learning_rate=2e-5,
            num_train_epochs=3,
            weight_decay=0.01,
            warmup_ratio=0.0,
            batch_size_override=16,
            dataset_name=DATASET_NAME,
            seed=SEED,
            run_transformer=RUN_TRANSFORMER,
            wandb_enabled=WANDB_ACTIVE,
            wandb_run_name=getattr(wandb_run, "name", None),
            output_subdir=f"transformer_configured_{safe_variant_key(model_name)}",
            evaluate_test=True,
            early_stopping_patience=1,
            save_total_limit=1,
        )
        transformer_outputs.append({"stage": "configured", "model_name": model_name, "output": output})

# 2) Run full two-stage transformer HPT for every transformer variant.
if RUN_TRANSFORMER_HPT:
    for model_name in TRANSFORMER_HPT_MODEL_VARIANTS:
        print(f"\n=== Transformer HPT variant: {model_name} ===")
        hpt_output = run_two_stage_transformer_hpt(
            train_df,
            validation_df,
            test_df,
            id2label,
            paths.results_dir,
            dataset_name=DATASET_NAME,
            config=TransformerHPTConfig(
                model_name=model_name,
                random_trials=HPT_RANDOM_TRIALS,
                bayes_trials=HPT_BAYES_TRIALS,
                seed=SEED,
                early_stopping_patience=1,
                save_total_limit=1,
                final_retrain=True,
                final_retrain_epochs=HPT_FINAL_RETRAIN_EPOCHS,
                max_train_samples=HPT_MAX_TRAIN_SAMPLES,
                max_validation_samples=HPT_MAX_VALIDATION_SAMPLES,
                max_eval_samples=HPT_MAX_EVAL_SAMPLES,
                smoke_test=TRANSFORMER_HPT_SMOKE_TEST,
            ),
            wandb_enabled=WANDB_ACTIVE,
            wandb_project=WANDB_PROJECT,
            wandb_entity=WANDB_ENTITY,
            wandb_mode=WANDB_MODE,
        )
        transformer_hpt_outputs[model_name] = hpt_output
        output = hpt_output.get("final_output") or {
            "result": None,
            "predictions": pd.DataFrame(),
            "trainer": None,
            "skip_result": {
                "model_family": "transformer",
                "model_name": model_name,
                "training_type": "fine-tuned supervised HPT",
                "dataset": DATASET_NAME,
                "eval_split": "test",
                "sample_size": 0,
                "accuracy": np.nan,
                "macro_f1": np.nan,
                "weighted_f1": np.nan,
                "notes": f"Transformer HPT did not produce a final model: {hpt_output.get('reason', 'unknown')}",
            },
        }
        transformer_outputs.append({"stage": "hpt_final", "model_name": model_name, "output": output, "hpt_output": hpt_output})

        # Preserve the module's standard final transformer directory before the next HPT variant overwrites it.
        standard_final_dir = paths.results_dir / "transformer"
        archive_final_dir = paths.results_dir / f"transformer_hpt_final_{safe_variant_key(model_name)}"
        if standard_final_dir.exists():
            if archive_final_dir.exists():
                shutil.rmtree(archive_final_dir)
            shutil.copytree(standard_final_dir, archive_final_dir)

        hpt_results_df = hpt_output.get("results")
        print(f"Transformer HPT status: {hpt_output.get('status')} | reason: {hpt_output.get('reason', '')}")
        print(f"Transformer HPT run root: {hpt_output.get('run_root')}")
        if isinstance(hpt_results_df, pd.DataFrame) and not hpt_results_df.empty:
            display(hpt_results_df[[
                "stage", "trial_number", "status", "validation_macro_f1", "learning_rate",
                "batch_size", "epochs", "weight_decay", "warmup_ratio", "max_length",
                "selected_for_final", "reason",
            ]])
elif not RUN_CONFIGURED_TRANSFORMER_BASELINES:
    print("No transformer baselines configured. Set RUN_CONFIGURED_TRANSFORMER_BASELINES or RUN_TRANSFORMER_HPT to True.")


In [ ]:
# @title
# Aggregate all configured/HPT transformer variants into the final comparison state.
for entry in transformer_outputs:
    stage = entry["stage"]
    model_name = entry["model_name"]
    output = entry["output"]
    result = output.get("result")
    skip_result = output.get("skip_result")
    result_stage_name = f"{safe_variant_key(model_name)}_{stage}"

    if result is not None:
        result_copy = dict(result)
        result_copy["model_name"] = result_stage_name
        result_copy["notes"] = f"Transformer variant={model_name}; stage={stage}. " + str(result_copy.get("notes", ""))
        completed_results.append(result_copy)

        pred_df = output.get("predictions", pd.DataFrame()).copy()
        if not pred_df.empty:
            pred_df["model_name"] = result_stage_name
            prediction_tables[result_stage_name] = pred_df

        if model_name == TRANSFORMER_MODEL_NAME and stage == "hpt_final":
            transformer_output = output
            trained_models["transformer_trainer"] = output.get("trainer")

        display(pd.DataFrame([result_copy])[["model_name", "accuracy", "macro_f1", "weighted_f1", "notes"]])

    elif skip_result is not None:
        skip_copy = dict(skip_result)
        skip_copy["model_name"] = result_stage_name
        skip_copy["notes"] = f"Transformer variant={model_name}; stage={stage}. " + str(skip_copy.get("notes", ""))
        completed_results.append(skip_copy)

if transformer_output.get("result") is None and transformer_outputs:
    # Fallback for downstream cells: use the first successful transformer output if the main HPT variant did not complete.
    for entry in transformer_outputs:
        if entry["output"].get("result") is not None:
            transformer_output = entry["output"]
            break


In [ ]:
# @title
finish_wandb_stage()  # end 08_transformers_configured_and_hpt_all_variants


## 9. Qwen2.5-Instruct Prompting Baseline

This stage optionally evaluates an instruction-tuned language model as a prompting baseline. Qwen is not fine-tuned; it is only prompted to classify clauses into the fixed LEDGAR label set.

Prompting setup:

| Mode | Description |
|---|---|
| Zero-shot | Provides the clause text and full list of allowed labels. |
| Static few-shot | Adds training examples only; validation and test examples are never used as demonstrations. |
| Retrieval few-shot | Retrieves similar examples from the training split only. |

Generation and parsing controls:

| Setting | Value |
|---|---:|
| Models | `Qwen/Qwen2.5-3B-Instruct`, `Qwen/Qwen2.5-7B-Instruct` |
| Evaluation sample | full test split requested; no cap (`QWEN_MAX_EVAL_SAMPLES=None`) |
| Decoding | deterministic, `do_sample=False` |
| New tokens | `max_new_tokens=20` |
| Output requirement | return exactly one allowed label |
| Parser | exact allowed-label match after whitespace/case normalisation |
| Invalid outputs | marked as `INVALID_PREDICTION` and reported separately |

Governance note: this is not directly equivalent to supervised fine-tuning. The prompted model has different pretraining and task setup, so results should be interpreted as a separate baseline rather than a perfectly fair model-family comparison.


In [ ]:
# @title
start_wandb_stage("09_qwen_prompting_all_variants", {
    "stage_type": "qwen_prompting_all_variants",
    "run_qwen": RUN_QWEN_BASELINE,
    "qwen_model_variants": QWEN_MODEL_VARIANTS,
})


In [ ]:
# @title
qwen_outputs = []
qwen_result_rows = []
qwen_prediction_frames = []
qwen_invalid_frames = []
qwen_model = None
qwen_tokenizer = None

if RUN_QWEN_BASELINE:
    for idx, model_name in enumerate(QWEN_MODEL_VARIANTS):
        print(f"\n=== Qwen prompting variant: {model_name} ===")
        current_output = run_qwen_baseline(
            train_df,
            test_df,
            label2id,
            id2label,
            paths.results_dir,
            model_name=model_name,
            label_names=label_names,
            eval_sample_size=QWEN_EVAL_SAMPLE_SIZE,
            few_shot_examples_per_class=QWEN_FEW_SHOT_EXAMPLES_PER_CLASS,
            dataset_name=DATASET_NAME,
            seed=SEED,
            run_qwen=True,
            max_eval_samples=QWEN_MAX_EVAL_SAMPLES,
            smoke_test=False,
        )

        # Preserve the module's standard qwen output directory before the next variant overwrites it.
        standard_qwen_dir = paths.results_dir / "qwen"
        archive_qwen_dir = paths.results_dir / f"qwen_{safe_variant_key(model_name)}"
        if standard_qwen_dir.exists():
            if archive_qwen_dir.exists():
                shutil.rmtree(archive_qwen_dir)
            shutil.copytree(standard_qwen_dir, archive_qwen_dir)

        model_key = safe_variant_key(model_name)
        for result in current_output.get("results", []):
            result_copy = dict(result)
            result_copy["model_name"] = f"{model_key}_{result_copy.get('model_name', 'qwen')}"
            result_copy["notes"] = f"Qwen variant={model_name}. " + str(result_copy.get("notes", ""))
            result_copy["evidence_archive_dir"] = str(archive_qwen_dir)
            qwen_result_rows.append(result_copy)
            completed_results.append(result_copy)

        pred_df = current_output.get("predictions", pd.DataFrame()).copy()
        if not pred_df.empty:
            pred_df["model_variant"] = model_name
            pred_df["model_name"] = pred_df["model_name"].apply(lambda x: f"{model_key}_{x}")
            qwen_prediction_frames.append(pred_df)

        invalid_df = current_output.get("invalid_outputs", pd.DataFrame()).copy()
        if not invalid_df.empty:
            invalid_df["model_variant"] = model_name
            invalid_df["model_name"] = invalid_df["model_name"].apply(lambda x: f"{model_key}_{x}")
            qwen_invalid_frames.append(invalid_df)

        # Keep only the final loaded Qwen model for the small agentic review stage to avoid holding multiple large models in memory.
        is_last_variant = idx == len(QWEN_MODEL_VARIANTS) - 1
        if is_last_variant:
            qwen_model = current_output.get("model")
            qwen_tokenizer = current_output.get("tokenizer")
        else:
            try:
                del current_output["model"]
                del current_output["tokenizer"]
                gc.collect()
                import torch
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
            except Exception:
                pass

        qwen_outputs.append({"model_name": model_name, "output": current_output})
else:
    print("Qwen prompting variants skipped because RUN_QWEN_BASELINE=False.")
    qwen_result_rows = []


In [ ]:
# @title
qwen_output = {"results": qwen_result_rows, "predictions": pd.DataFrame(), "invalid_outputs": pd.DataFrame(), "model": qwen_model, "tokenizer": qwen_tokenizer}
qwen_predictions_df = pd.concat(qwen_prediction_frames, ignore_index=True) if qwen_prediction_frames else pd.DataFrame()
qwen_invalid_outputs_df = pd.concat(qwen_invalid_frames, ignore_index=True) if qwen_invalid_frames else pd.DataFrame()
qwen_output["predictions"] = qwen_predictions_df
qwen_output["invalid_outputs"] = qwen_invalid_outputs_df

if qwen_output["results"]:
    display(pd.DataFrame(qwen_output["results"])[["model_name", "accuracy", "macro_f1", "weighted_f1", "notes"]])
else:
    print("No completed Qwen prompting results were added.")


In [ ]:
# @title
finish_wandb_stage()  # end 09_qwen_prompting_all_variants


## 12. Final Model Comparison

This stage consolidates all completed and skipped model runs into one comparison table. It does not insert or fabricate any metrics; it only formats rows produced by earlier sections.

Comparison columns:

| Column | Meaning |
|---|---|
| `model_family` | baseline, classical, transformer, or prompting family. |
| `model_name` | specific model/configuration name. |
| `training_type` | dummy, supervised, fine-tuned, prompted, or skipped. |
| `dataset` | evaluation dataset, here LEDGAR for the main experiment. |
| `eval_split` | split used for reported metrics, usually test. |
| `sample_size` | number of evaluated examples. |
| `accuracy` | overall exact-label accuracy. |
| `macro_f1` | unweighted mean F1 across classes; primary metric. |
| `weighted_f1` | class-frequency-weighted F1. |
| `notes` | skip reason or relevant run detail. |

The table and macro-F1 plot are saved under `results/` for later inspection.


In [ ]:
# @title
start_wandb_stage("12_final_comparison", {"stage_type": "final_comparison"})


In [ ]:
# @title
comparison_df = save_final_comparison(completed_results, paths.results_dir)

print(f"Saved final comparison to: {paths.results_dir / 'final_model_comparison.csv'}")

display(comparison_df)

In [ ]:
# @title
finish_wandb_stage()  # end 12_final_comparison


## 13. Error Analysis

This stage checks model behaviour beyond aggregate metrics. It uses saved predictions, confusion outputs, and class counts from earlier stages.

Outputs reviewed:

| Output | Purpose |
|---|---|
| Top confused label pairs | Shows where the best classical model mixes labels. |
| Misclassified examples | Provides examples for qualitative inspection. |
| Transformer errors | Included only when the transformer ran. |
| Qwen invalid outputs | Included only when Qwen produced real predictions. |
| Class imbalance summary | Shows how training labels are distributed. |

These outputs support later discussion without writing report conclusions here.


In [ ]:
# @title
start_wandb_stage("13_error_analysis", {"stage_type": "error_analysis"})


In [ ]:
# @title
error_outputs = run_error_analysis(
    comparison_df,
    prediction_tables,
    train_df,
    paths.results_dir,
    best_classical_name=best_classical_name,
    transformer_model_name=TRANSFORMER_MODEL_NAME,
    qwen_predictions_df=qwen_predictions_df,
    qwen_invalid_outputs_df=qwen_invalid_outputs_df,
)


Interpretation focus:

| Issue | Why it matters |
|---|---|
| Class imbalance | Accuracy can look high while minority classes perform poorly. |
| Label ambiguity | Legal clauses may plausibly fit more than one clause type. |
| Long clauses | Transformer truncation and TF-IDF sparsity can affect predictions. |
| Boilerplate wording | Repeated legal phrasing can make labels harder to separate. |
| Invalid LLM outputs | Prompted models may ignore the closed label set. |

In [ ]:
print(f"Best completed model by macro-F1: {error_outputs.get('best_model_name')}")

In [ ]:
# @title
if "classical_confusions" in error_outputs:
    print("Top classical confused label pairs:")
    display(error_outputs["classical_confusions"])

if "classical_misclassified" in error_outputs:
    print("Classical misclassified examples:")
    display(error_outputs["classical_misclassified"][["text", "label", "predicted_label"]])

if "transformer_misclassified" in error_outputs:
    print("Transformer misclassified examples:")
    display(error_outputs["transformer_misclassified"][["text", "label", "predicted_label"]])

if "qwen_invalid_outputs" in error_outputs:
    print("Qwen invalid outputs:")
    display(error_outputs["qwen_invalid_outputs"].head(10))

if "qwen_plausible_nonmatching" in error_outputs:
    print("Qwen plausible non-matching label examples for manual inspection:")
    display(error_outputs["qwen_plausible_nonmatching"].head(10))

print("Class imbalance summary:")
display(error_outputs.get("class_imbalance", pd.DataFrame()).head(20))


In [ ]:
# @title
finish_wandb_stage()  # end 13_error_analysis
